# MF2 60x Sil 1.3 — Chromatic Z-correction

Measures the axial (Z) and lateral (XY) offset between color channels using
0.1 µm TetraSpek beads imaged through a 0 – 30 µm Z-stack.

**Part 1** — Generates the HAL config and shutter files for the calibration acquisition.  
**Part 2** — Reads the resulting DAX files, localises beads in 3D by Gaussian fitting,
and plots the average displacement of each channel relative to 405 nm.

**Inputs (Part 2):** one or more `.dax` files + matching `.inf` sidecars from the
calibration acquisition, and the `frame_table_*.csv` saved in Part 1.  
**Output:** `zcorrection_beads.csv` with columns
`bead_id, fov_id, 405_x, 405_y, 405_z, 488_x, 488_y, 488_z, 560_x, 560_y, 560_z,
650_x, 650_y, 650_z` (all in µm).

In [ ]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.ndimage  import gaussian_filter, maximum_filter
from scipy.spatial  import KDTree

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/  (notebook lives in MERci/notebooks/misc/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.configs import (
    get_frame_table, get_color_sequence_name,
    create_shutter_file, create_hal_config,
)
from MERci.acquisition.display  import print_frame_table, display_xml
from MERci.common.io            import parse_inf
from MERci.visualization        import visualize_shutter_sequence

logging.basicConfig(level=logging.WARNING)
print(f"MERCI_DIR  : {MERCI_DIR}")
print(f"SAMPLE_DIR : {SAMPLE_DIR}")

---
## Part 0 — Pipeline validation using simulated data

Before running on real data, this part validates the full localisation pipeline
using a synthetic TetraSpek-bead volume.

**No chromatic aberration test**: simulate 4-colour stacks from identical emitter
positions. Recovered shifts should be ΔX = ΔY = ΔZ ≈ 0 for all channels.

**Known chromatic aberration test**: apply known Z-shifts (0, 0.5, 1.0, 1.5 µm)
and verify the pipeline recovers them within fitting uncertainty.

*Run Part 0 first, before the acquisition. It catches configuration errors
and gives a sense of the fitting precision for this PSF size and SNR.*

In [ ]:
from MERci.analysis.spot_localization import (
    generate_emitter_positions,
    simulate_multicolor_stack,
    localize_beads_in_volume,
    match_beads_across_colors,
    plot_max_projections,
)

# ── Simulation parameters ──────────────────────────────────────────────
SIM_N_EMITTERS  = 20
SIM_PIXEL_SIZE  = 0.109          # µm/px  (MF2 60x silicone 1.3 NA)
SIM_Z_STEP      = 0.2            # µm/plane
SIM_PSF_XY_UM   = 0.15           # lateral PSF sigma in µm (~1.4 px)
SIM_PSF_Z_UM    = 0.35           # axial PSF sigma in µm (~1.75 z-steps)
SIM_PHOTONS     = 2000.0         # peak photons per emitter (after PSF convolution)
SIM_READOUT     = 5.0            # Gaussian readout noise std (ADU, typical sCMOS)
SIM_BG          = 50.0           # uniform background (photons)
SIM_COLORS      = [405, 488, 560, 650]
SIM_Z_POS       = np.arange(0, 30.2, 0.2)       # 151 planes — same as real acquisition
SIM_N_Y, SIM_N_X = 200, 200                     # small FOV for speed
SIM_VOL_SHAPE   = (len(SIM_Z_POS), SIM_N_Y, SIM_N_X)
SIM_VOXEL       = (SIM_PIXEL_SIZE, SIM_PIXEL_SIZE, SIM_Z_STEP)

# Localisation settings (same as Part 2)
SIM_CROP_XY      = 10
SIM_CROP_Z       = 12
SIM_MIN_DIST_PX  = 15
SIM_THRESH_SIGMA = 3.0
SIM_MATCH_TOL_PX = 5.0

# ── Generate emitter positions ─────────────────────────────────────────
rng = np.random.default_rng(42)
sim_positions = generate_emitter_positions(
    volume_um     = (SIM_N_X * SIM_PIXEL_SIZE, SIM_N_Y * SIM_PIXEL_SIZE, float(SIM_Z_POS[-1])),
    voxel_size_um = SIM_VOXEL,
    n_emitters    = SIM_N_EMITTERS,
    rng           = rng,
)

# ── Simulate 4-colour stack (no chromatic aberration) ─────────────────
sim_stacks = simulate_multicolor_stack(
    sim_positions, SIM_COLORS, SIM_Z_POS, SIM_VOL_SHAPE, SIM_VOXEL,
    SIM_PSF_XY_UM, SIM_PSF_Z_UM,
    photon_budget     = SIM_PHOTONS,
    readout_noise_std = SIM_READOUT,
    bg_photons        = SIM_BG,
    rng               = rng,
)

# ── Visualise 405 nm projections ───────────────────────────────────────
# White = high intensity (beads appear as bright spots on dark background)
fig = plot_max_projections(sim_stacks[405], SIM_VOXEL, title="Simulated 405 nm — no chromatic aberration")
plt.show()

# ── Localise beads in all channels ────────────────────────────────────
print("Localising simulated beads (no chromatic aberration)...")
sim_color_dfs = localize_beads_in_volume(
    sim_stacks, SIM_Z_POS,
    crop_xy      = SIM_CROP_XY,
    crop_z       = SIM_CROP_Z,
    min_dist_px  = SIM_MIN_DIST_PX,
    thresh_sigma = SIM_THRESH_SIGMA,
)
for color, df in sim_color_dfs.items():
    print(f"  {color} nm : {len(df)} beads fitted")

# ── Match across channels and report shifts ────────────────────────────
sim_beads = match_beads_across_colors(
    sim_color_dfs, ref_color=405,
    match_tol_px=SIM_MATCH_TOL_PX, pixel_size_um=SIM_PIXEL_SIZE,
)
print(f"\nMatched {len(sim_beads)} beads across all channels")
print("Expected result: all shifts ~0 (no chromatic aberration)\n")
for color in [488, 560, 650]:
    if f"{color}_z" not in sim_beads.columns:
        continue
    dx = sim_beads[f"{color}_x"] - sim_beads["405_x"]
    dy = sim_beads[f"{color}_y"] - sim_beads["405_y"]
    dz = sim_beads[f"{color}_z"] - sim_beads["405_z"]
    print(f"  {color} nm  DX={dx.mean():+.4f}±{dx.std():.4f}  "
          f"DY={dy.mean():+.4f}±{dy.std():.4f}  "
          f"DZ={dz.mean():+.4f}±{dz.std():.4f}  µm")

In [ ]:
# ── Part 0e — Chromatic aberration test ──────────────────────────────────────
# Simulate the same volume but with known Z-shifts applied to each colour.
# The recovered shifts should match KNOWN_SHIFTS_UM within fitting uncertainty.

KNOWN_SHIFTS_UM = {
    405: (0.0, 0.0, 0.0),    # reference: no shift
    488: (0.0, 0.0, 0.5),    # 0.5 µm axial shift
    560: (0.0, 0.0, 1.0),    # 1.0 µm axial shift
    650: (0.0, 0.0, 1.5),    # 1.5 µm axial shift
}

rng_ca    = np.random.default_rng(42)
stacks_ca = simulate_multicolor_stack(
    sim_positions, SIM_COLORS, SIM_Z_POS, SIM_VOL_SHAPE, SIM_VOXEL,
    SIM_PSF_XY_UM, SIM_PSF_Z_UM,
    color_shifts_um   = KNOWN_SHIFTS_UM,
    photon_budget     = SIM_PHOTONS,
    readout_noise_std = SIM_READOUT,
    bg_photons        = SIM_BG,
    rng               = rng_ca,
)

ca_color_dfs = localize_beads_in_volume(
    stacks_ca, SIM_Z_POS,
    crop_xy=SIM_CROP_XY, crop_z=SIM_CROP_Z,
    min_dist_px=SIM_MIN_DIST_PX, thresh_sigma=SIM_THRESH_SIGMA,
)
ca_beads = match_beads_across_colors(
    ca_color_dfs, ref_color=405,
    match_tol_px=SIM_MATCH_TOL_PX, pixel_size_um=SIM_PIXEL_SIZE,
)

print(f"Matched {len(ca_beads)} beads (chromatic aberration test)")
print(f"\n{'Channel':>10}  {'Measured DZ (µm)':>18}  {'Expected DZ (µm)':>18}  {'Error (µm)':>12}")
print("-" * 66)
for color in [488, 560, 650]:
    if f"{color}_z" not in ca_beads.columns:
        continue
    dz_measured = (ca_beads[f"{color}_z"] - ca_beads["405_z"]).mean()
    dz_expected = KNOWN_SHIFTS_UM[color][2]
    print(f"{color:>10} nm  {dz_measured:>+18.4f}  {dz_expected:>+18.3f}  {dz_measured - dz_expected:>+12.4f}")

---
## Part 1 — Create HAL config for the calibration acquisition

Generates the shutter and HAL config XML files for a 4-colour, 0 – 30 µm Z-stack
on MF2 using DAX format.  Run this **before** the acquisition.

In [ ]:
SETTINGS_DIR = SAMPLE_DIR / "settings"
METADATA_DIR = SAMPLE_DIR / "metadata"
SETTINGS_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

MICROSCOPE    = "MF2"
FILE_TYPE     = ".dax"
EXPOSURE_TIME = 0.3      # seconds

# ── Auto-detect HAL template ───────────────────────────────────────────
_hal_dir        = MERCI_DIR / "data" / "configs" / "hal"
_hal_candidates = sorted(
    p for p in _hal_dir.glob("hal-config-*.xml")
    if MICROSCOPE.lower() in p.name.lower()
)
if not _hal_candidates:
    raise FileNotFoundError(
        f"No HAL template found for microscope '{MICROSCOPE}' in {_hal_dir}"
    )
HAL_TEMPLATE = _hal_candidates[0]

# ── Z-stack definition ─────────────────────────────────────────────────
# z_bead is set to -1 µm (outside the data range) so the Z-nanopositioner
# has a reference plane that does not overlap with the bead data stack.
z_bead    = -1.0
bead_seq  = [np.nan]              # one blank (dark) frame as Z reference
color_seq = [405, 488, 560, 650]  # interleaved: all 4 colours per z-plane
end_seq   = [np.nan]              # blank frame to allow Z-nanopositioner return
z_pos     = np.arange(0, 30.2, 0.2)   # 0.0, 0.2, …, 30.0 µm  (151 planes)
SCAN_MODE = "interleaved"

print(f"HAL template : {HAL_TEMPLATE.name}")
print(f"Z-planes     : {len(z_pos)}  ({z_pos[0]:.1f} – {z_pos[-1]:.1f} µm)")
print(f"Total frames : {1 + len(z_pos) * len(color_seq) + 1}  "
      f"(1 bead + {len(z_pos)}×{len(color_seq)} data + 1 end)")

In [ ]:
frame_table = get_frame_table(
    z_bead, bead_seq, color_seq, end_seq, z_pos,
    microscope=MICROSCOPE, scan_mode=SCAN_MODE,
)
name = get_color_sequence_name(frame_table, scan_mode=SCAN_MODE)
print(f"Config name : {name}")
print_frame_table(frame_table)

# ── Save frame table ───────────────────────────────────────────────────
ft_path = METADATA_DIR / f"frame_table_{name}.csv"
frame_table.to_csv(ft_path)
print(f"Frame table : {ft_path}")

# ── Write shutter file ─────────────────────────────────────────────────
shutter_name = f"shutter-{name}.xml"
shutter_path = SETTINGS_DIR / shutter_name
create_shutter_file(frame_table, shutter_path)
print(f"Shutter     : {shutter_path}")
display_xml(shutter_path)

# ── Write HAL config ───────────────────────────────────────────────────
hal_name   = f"hal-config-{MICROSCOPE.lower()}-zcal-{name}.xml"
hal_output = SETTINGS_DIR / hal_name
create_hal_config(
    HAL_TEMPLATE, frame_table, shutter_name, hal_output,
    file_type=FILE_TYPE, exposure_time=EXPOSURE_TIME,
)
print(f"HAL config  : {hal_output}")
display_xml(hal_output)

In [ ]:
visualize_shutter_sequence(
    frame_table,
    title=f"Shutter sequence: {name}",
    save_path=METADATA_DIR / f"shutter_sequence_{name}.png",
)

---
## Part 2 — Bead localisation and chromatic shift analysis

Reads a list of DAX files, detects TetraSpek beads in each colour channel via
3D Gaussian fitting, and measures the average XYZ displacement of 488, 560, and
650 nm relative to the 405 nm reference channel.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
# Edit DAX_FILES to point to the files acquired with the config above.
DAX_FILES = [
    SAMPLE_DIR / "data" / "zcal_fov001.dax",
    # add more files as needed
]

# Frame table produced in Part 1 (auto-detected from metadata/ if name is defined)
try:
    FRAME_TABLE_CSV = ft_path       # if Part 1 was run in the same session
except NameError:
    FRAME_TABLE_CSV = sorted(METADATA_DIR.glob("frame_table_405f*-488f*.csv"))[-1]

OUTPUT_CSV = SAMPLE_DIR / "metadata" / "zcorrection_beads.csv"

# ── Acquisition constants ──────────────────────────────────────────────
PIXEL_SIZE_UM = 0.109   # µm / pixel  (MF2 with 60x silicone 1.3 NA)
Z_STEP_UM     = 0.2     # µm per z-plane
REF_COLOR     = 405     # reference channel for shift measurement

# ── Bead detection ─────────────────────────────────────────────────────
DETECT_MIN_DIST_PX  = 15    # minimum centre-to-centre separation (pixels)
DETECT_THRESH_SIGMA = 3.0   # detection threshold above background (sigma)

# ── 3D Gaussian crop ───────────────────────────────────────────────────
CROP_XY_PX = 10   # half-width of XY crop around each bead centre (pixels)
CROP_Z_PL  = 12   # half-width of Z crop around the rough z-peak (planes, 2.4 µm)

# ── Bead matching ──────────────────────────────────────────────────────
MATCH_TOL_PX = 5.0  # maximum XY distance (pixels) to match a bead across colours

print(f"Frame table : {FRAME_TABLE_CSV.name}")
print(f"DAX files   : {len(DAX_FILES)}")

In [ ]:
from MERci.analysis.spot_localization import (
    localize_beads_in_file,
    match_beads_across_colors,
)

In [ ]:
# ── Main analysis loop ─────────────────────────────────────────────────

frame_table = pd.read_csv(FRAME_TABLE_CSV, index_col=0)

all_results = []
for fov_id, dax_path in enumerate(DAX_FILES):
    print(f"\nFOV {fov_id}  —  {dax_path.name}")

    color_dfs = localize_beads_in_file(
        dax_path, frame_table,
        crop_xy=CROP_XY_PX, crop_z=CROP_Z_PL,
        min_dist_px=DETECT_MIN_DIST_PX, thresh_sigma=DETECT_THRESH_SIGMA,
    )

    df_fov = match_beads_across_colors(
        color_dfs, ref_color=REF_COLOR,
        match_tol_px=MATCH_TOL_PX, pixel_size_um=PIXEL_SIZE_UM,
    )
    df_fov.insert(1, "fov_id", fov_id)
    all_results.append(df_fov)
    print(f"  {len(df_fov)} beads matched across all channels")

beads_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# Re-number bead_id globally
beads_df["bead_id"] = np.arange(len(beads_df))

beads_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nTotal beads : {len(beads_df)}")
print(f"Saved       : {OUTPUT_CSV}")
beads_df.head()

---
## Part 3 — Chromatic shift visualisation

Average displacement of each channel relative to 405 nm (reference).
Error bars show the standard deviation across all beads.

In [ ]:
# ── Bar charts: mean shift per channel ────────────────────────────────

compare_colors = [c for c in [488, 560, 650] if f"{c}_x" in beads_df.columns]

shift_data = []
for color in compare_colors:
    dx = beads_df[f"{color}_x"] - beads_df[f"{REF_COLOR}_x"]
    dy = beads_df[f"{color}_y"] - beads_df[f"{REF_COLOR}_y"]
    dz = beads_df[f"{color}_z"] - beads_df[f"{REF_COLOR}_z"]
    shift_data.append({
        "color": color,
        "dx_mean": dx.mean(), "dx_std": dx.std(),
        "dy_mean": dy.mean(), "dy_std": dy.std(),
        "dz_mean": dz.mean(), "dz_std": dz.std(),
    })

shift_df = pd.DataFrame(shift_data)
print(shift_df.to_string(index=False))

_COLOR_HEX = {488: "#1f77b4", 560: "#ff7f0e", 650: "#2ca02c"}
x_pos = np.arange(len(compare_colors))
xlabels = [f"{c} nm" for c in compare_colors]

fig, axes = plt.subplots(1, 3, figsize=(11, 4), sharey=False)
for ax, axis_label, mean_col, std_col in zip(
    axes,
    ["\u0394X (µm)", "\u0394Y (µm)", "\u0394Z (µm)"],
    ["dx_mean", "dy_mean", "dz_mean"],
    ["dx_std",  "dy_std",  "dz_std"],
):
    colors_hex = [_COLOR_HEX.get(c, "#7f7f7f") for c in compare_colors]
    ax.bar(x_pos, shift_df[mean_col], yerr=shift_df[std_col],
           color=colors_hex, edgecolor="k", linewidth=0.8,
           capsize=5, error_kw={"linewidth": 1.2})
    ax.axhline(0, color="k", linewidth=0.8, linestyle="--")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(xlabels)
    ax.set_ylabel(axis_label)
    ax.set_title(f"Mean {axis_label} vs {REF_COLOR} nm")

fig.suptitle(
    f"Chromatic shift  |  {len(beads_df)} beads, {len(DAX_FILES)} FOV(s)",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
fig.savefig(
    SAMPLE_DIR / "metadata" / "zcorrection_shifts.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

In [ ]:
# ── Optional: ΔZ vs z_405 — detects field-dependent z-curvature ───────

fig, axes = plt.subplots(1, len(compare_colors),
                         figsize=(5 * len(compare_colors), 4), sharey=False)
if len(compare_colors) == 1:
    axes = [axes]

for ax, color in zip(axes, compare_colors):
    z_ref = beads_df[f"{REF_COLOR}_z"]
    dz    = beads_df[f"{color}_z"] - z_ref
    ax.scatter(z_ref, dz, s=12, alpha=0.5, color=_COLOR_HEX.get(color, "#7f7f7f"))
    # linear trend
    if len(z_ref) > 1:
        coeff = np.polyfit(z_ref, dz, 1)
        z_line = np.linspace(z_ref.min(), z_ref.max(), 100)
        ax.plot(z_line, np.polyval(coeff, z_line), "k--", linewidth=1.2)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    ax.set_xlabel(f"z_405 (µm)")
    ax.set_ylabel(f"\u0394Z  {color} \u2013 405 (µm)")
    ax.set_title(f"{color} nm  vs  {REF_COLOR} nm")

fig.suptitle("Z-shift vs reference z position", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(
    SAMPLE_DIR / "metadata" / "zcorrection_zdependent.png",
    dpi=200, bbox_inches="tight",
)
plt.show()